In [ ]:
import os
from typing import TypedDict, List
from pydantic import BaseModel
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [ ]:
class AgentState(TypedDict):
    evidence: str
    features_data: str
    setup_data: str
    readme: str

class FeatureSchema(BaseModel):
    features: List[str]

class SetupSchema(BaseModel):
    commands: List[str]

In [ ]:
def feature_agent(state: AgentState):
    model = llm.with_structured_output(FeatureSchema)
    res = model.invoke(f"Find features in: {state['evidence']}")
    return {"features_data": str(res.features)}

def setup_agent(state: AgentState):
    model = llm.with_structured_output(SetupSchema)
    res = model.invoke(f"Find setup commands in: {state['evidence']}")
    return {"setup_data": str(res.commands)}

def coordinator(state: AgentState):
    prompt = ChatPromptTemplate.from_template("Write README. Features: {f}, Setup: {s}")
    res = (prompt | llm).invoke({"f": state["features_data"], "s": state["setup_data"]})
    return {"readme": res.content}

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("start", lambda x: x)
workflow.add_node("feature_agent", feature_agent)
workflow.add_node("setup_agent", setup_agent)
workflow.add_node("coordinator", coordinator)

workflow.set_entry_point("start")
workflow.add_edge("start", "feature_agent")
workflow.add_edge("start", "setup_agent")
workflow.add_edge("feature_agent", "coordinator")
workflow.add_edge("setup_agent", "coordinator")
workflow.add_edge("coordinator", END)

app = workflow.compile()

In [ ]:
state = {
    "evidence": "This is a python fastAPI app. It has RAG features and multi-agent workflow. Run pip install -r requirements.txt to install.",
    "features_data": "",
    "setup_data": "",
    "readme": ""
}

result = app.invoke(state)
print(result["readme"])